# Grenzen der Linearen Regression

Dieses Notebook ist kurz, wird aber hoffentlich die Grenzen der linearen Regression verdeutlichen. Ein lineares Regressionsmodell ist nur in der Lage, **lineare** Beziehungen zu identifizieren.
Wir können das berühmte [Anscombe-Quartett](https://en.wikipedia.org/wiki/Anscombe%27s_quartet) Dataset verwenden, um zu zeigen, dass lineare Regression für sehr unterschiedliche Datensätze die gleichen Ergebnisse liefern kann.

In [ ]:
# Import required libraries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats 

%matplotlib inline

## Das Anscombe-Quartett

In [ ]:
# Anscombe dataset includes 4 different data sets
anscombe = sns.load_dataset("anscombe")

In [ ]:
# Get the descriptive statistics for each dataset
anscombe.groupby('dataset').describe()

Wie Sie aus der obigen Tabelle sehen können, haben alle vier Datensätze sehr ähnliche deskriptive Statistiken. Wenn wir uns nur diese Zusammenfassung ansehen würden, könnten wir fälschlicherweise annehmen, dass die Datensätze auch ziemlich ähnlich aussehen.
Lassen Sie uns nun einige zusammenfassende Statistiken berechnen, um die Daten mit ihren Regressionslinien zu plotten.
Eine einfache Möglichkeit, eine lineare Kleinstquadrate-Regression zu berechnen und die interessanten Statistiken zurückzugeben, ist die Verwendung der `.linregress()` Funktion aus dem `scipy.stats` Modul.

In [ ]:
# Defining function using linregress of stats module 
# for building a linear regression model for each dataset
# Returns summary statistics
def get_summarystats(dataset):
    slope, intercept, r_value, p_value, std_err = stats.linregress(x=anscombe[anscombe.dataset==dataset].x,
                                                               y=anscombe[anscombe.dataset==dataset].y)
    
    
    return slope, intercept, r_value, p_value, std_err

In [ ]:
# Getting summary statistics for each dataset
results = []
for dataset in 'I II III IV'.split():
    slope, intercept, r_value, p_value, std_err = get_summarystats(dataset)
    results.append([slope, intercept, r_value])

print('[slope, intercept, r_value] for each dataset')
results

Die linearen Regressionsparameter sind ebenfalls nahezu identisch. Die Datensätze sehen jedoch deutlich unterschiedlich aus, wenn wir sie mit ihren Regressionslinien plotten.

In [ ]:
# Show the results of a linear regression within each dataset
sns.lmplot(x="x", y="y", col="dataset", hue="dataset", data=anscombe,
            col_wrap=2, ci=None, palette='muted', height=4, scatter_kws={"s": 50, "alpha": 1});

Aus den obigen Diagrammen ist es offensichtlich, dass die angepassten Linien grundsätzlich identisch sind - obwohl wir völlig unterschiedliche Daten haben.
In diesen Fällen ist es absichtlich offensichtlich, dass das Modell Schwierigkeiten hat, Linien an die Datensätze II, III und IV anzupassen. Normalerweise ist es notwendig, sich das Diagramm der Residuen anzusehen, um diese Fehler zu erkennen.

## Residuendiagramme

In einem echten Data-Science-Projekt reicht es nicht aus, die Qualität Ihres Modells in einem Wert (wie r-quadrat, RMSE...) zusammenzufassen, Sie müssen tiefer graben und herausfinden, ob die Fehler bestimmten Mustern folgen.
Eine gängige Methode, dies zu analysieren, ist die Erstellung sogenannter **Residuendiagramme**. Diese Diagramme zeigen - wie der Name schon sagt - die Verteilung der Residuen Ihres Modells. Sie sind ein wesentlicher Bestandteil jeder Fehleranalyse bei Regressionsproblemen!

Wir können Residuendiagramme für jeden Datensatz mit der folgenden Funktion erstellen.

In [ ]:
# Defining function which calculates residuals and plots them
def get_residuals(dataset, color):
    # store true y values
    obs_values = anscombe[anscombe.dataset==dataset].y 
    # store predicted y values
    pred_values = get_summarystats(dataset)[0] * anscombe[anscombe.dataset==dataset].x + get_summarystats(dataset)[1] 
    # calculate residuals
    residuals = obs_values - pred_values
    # plot residuals
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(anscombe[anscombe.dataset==dataset].x, residuals, color=color)
    ax.set_ylabel("Residuals")
    ax.set_xlabel("x")
    fig.suptitle('Residual Scatter Plot')
    plt.show()

In [ ]:
# Define same color palette as above
color = sns.color_palette('muted')

# Use defined function to plot residuals for all datasets
for dataset, c in zip('I II III IV'.split(), color):
    get_residuals(dataset, c) 

Bei einem gut angepassten Modell sind die Residuen zufällig verteilt. In unserem Fall trifft dies nur auf Datensatz I zu. Bei eher schlecht angepassten Modellen finden Sie Muster in Ihrer Residuenverteilung, wie wir es für die Datensätze II, III und IV sehen können. Diese Muster zeigen uns, dass unserem Modell zusätzliche erklärende Faktoren fehlen.

### Weiterführende Literatur

Wenn Sie irgendwann mehr über Residuen/Residuendiagramme erfahren möchten, können Sie mit diesen Artikeln beginnen:

* [What are residuals in statistics?](https://www.statology.org/residuals/)
* [How to use residuals for regression model validation?](https://towardsdatascience.com/how-to-use-residual-plots-for-regression-model-validation-c3c70e8ab378)
* [Interpreting residual plots to improve your regression](https://www.qualtrics.com/support/stats-iq/analyses/regression-guides/interpreting-residual-plots-improve-regression/)